In [1]:
# ============================================================
# BirdCLEF+ 2026 | Cell 1: EDA — CORRECTED PATHS
# ============================================================
import os, warnings
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# ── CORRECTED BASE PATH ──────────────────────────────────────
BASE        = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_CSV   = BASE / 'train.csv'
TAXON_CSV   = BASE / 'taxonomy.csv'
SS_LABELS   = BASE / 'train_soundscapes_labels.csv'
SAMPLE_SUB  = BASE / 'sample_submission.csv'
TRAIN_AUDIO = BASE / 'train_audio'
TRAIN_SS    = BASE / 'train_soundscapes'

# ── Load ─────────────────────────────────────────────────────
train      = pd.read_csv(TRAIN_CSV)
taxon      = pd.read_csv(TAXON_CSV)
ss_labels  = pd.read_csv(SS_LABELS)
sample_sub = pd.read_csv(SAMPLE_SUB)

print("=" * 60)
print("1. FILE SHAPES")
print("=" * 60)
print(f"train.csv          : {train.shape}")
print(f"taxonomy.csv       : {taxon.shape}")
print(f"ss_labels.csv      : {ss_labels.shape}")
print(f"sample_submission  : {sample_sub.shape}")
print(f"\ntrain.csv columns  : {train.columns.tolist()}")
print(f"\ntrain.csv sample:\n{train.head(3).to_string()}")

# ── Taxonomy ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. TAXONOMY BREAKDOWN")
print("=" * 60)
print(taxon['class_name'].value_counts())
print(f"\nTaxon columns: {taxon.columns.tolist()}")
print(f"\nSample rows:\n{taxon.head(8).to_string()}")

# ── Class imbalance ──────────────────────────────────────────
print("\n" + "=" * 60)
print("3. CLASS IMBALANCE")
print("=" * 60)
label_counts = train['primary_label'].value_counts()
print(f"Total clips          : {len(train)}")
print(f"Unique species       : {train['primary_label'].nunique()}")
print(f"Mean clips/species   : {label_counts.mean():.1f}")
print(f"Median clips/species : {label_counts.median():.1f}")
print(f"Max clips            : {label_counts.max()} ({label_counts.idxmax()})")
print(f"Min clips            : {label_counts.min()} ({label_counts.idxmin()})")
print(f"\nSpecies with <  5 clips : {(label_counts <  5).sum()}")
print(f"Species with < 10 clips : {(label_counts < 10).sum()}")
print(f"Species with < 20 clips : {(label_counts < 20).sum()}")
print(f"Species with < 50 clips : {(label_counts < 50).sum()}")
print(f"\nBottom 20 (fewest clips):")
print(label_counts.tail(20).to_string())

# ── Coverage ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. COVERAGE — ZERO CLIP SPECIES")
print("=" * 60)
submission_species = [c for c in sample_sub.columns if c != 'row_id']
train_species      = set(train['primary_label'].unique())
taxon_species      = set(taxon['primary_label'].unique())

print(f"Species in submission  : {len(submission_species)}")
print(f"Species in train_audio : {len(train_species)}")

soundscape_only = taxon_species - train_species
print(f"\nSpecies with ZERO train_audio clips: {len(soundscape_only)}")
for sp in sorted(soundscape_only):
    row = taxon[taxon['primary_label'] == sp]
    cls = row['class_name'].values[0] if len(row) > 0 else '?'
    print(f"  {sp:30s} ({cls})")

# ── Soundscape labels ─────────────────────────────────────────
print("\n" + "=" * 60)
print("5. SOUNDSCAPE LABELS (IN-DOMAIN GOLD)")
print("=" * 60)
print(f"ss_labels shape   : {ss_labels.shape}")
print(f"ss_labels columns : {ss_labels.columns.tolist()}")
print(f"\nSample rows:\n{ss_labels.head(5).to_string()}")

ss_species_flat = []
for labels in ss_labels['primary_label'].dropna():
    ss_species_flat.extend(str(labels).split(';'))
ss_counts = Counter(ss_species_flat)
print(f"\nUnique species in soundscape labels : {len(ss_counts)}")
print(f"Top 20:")
for sp, cnt in ss_counts.most_common(20):
    print(f"  {sp:30s}: {cnt}")

# ── Sample weights ────────────────────────────────────────────
print("\n" + "=" * 60)
print("6. SAMPLE WEIGHT PLAN")
print("=" * 60)
wt = {}
for sp in taxon_species:
    n = label_counts.get(sp, 0)
    if sp in soundscape_only : wt[sp] = 5.0
    elif n < 5               : wt[sp] = 4.0
    elif n < 20              : wt[sp] = 2.0
    elif n < 50              : wt[sp] = 1.5
    else                     : wt[sp] = 1.0
wt_dist = Counter(wt.values())
for w in sorted(wt_dist, reverse=True):
    print(f"  weight {w:.1f} : {wt_dist[w]:3d} species")

# ── Audio stats ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("7. AUDIO DURATION STATS (150 files)")
print("=" * 60)
audio_files = list(TRAIN_AUDIO.rglob('*.ogg'))
print(f"Total .ogg files: {len(audio_files)}")

durations = []
for fp in audio_files[:150]:
    try:
        durations.append(librosa.get_duration(path=str(fp)))
    except: pass

if durations:
    d = np.array(durations)
    print(f"Mean   : {d.mean():.2f}s")
    print(f"Median : {np.median(d):.2f}s")
    print(f"Min    : {d.min():.2f}s")
    print(f"Max    : {d.max():.2f}s")
    print(f"< 5s   : {(d < 5).sum()}")
    print(f"5–30s  : {((d>=5)&(d<=30)).sum()}")
    print(f"> 30s  : {(d > 30).sum()}")

# ── Soundscape files ──────────────────────────────────────────
print("\n" + "=" * 60)
print("8. SOUNDSCAPE FILES")
print("=" * 60)
ss_files = list(TRAIN_SS.rglob('*.ogg'))
print(f"Train soundscape .ogg count: {len(ss_files)}")
print("Sample filenames:")
for f in ss_files[:6]: print(f"  {f.name}")

# ── Submission structure ──────────────────────────────────────
print("\n" + "=" * 60)
print("9. SUBMISSION STRUCTURE")
print("=" * 60)
print(f"Submission rows: {len(sample_sub)}")
sample_sub['soundscape'] = sample_sub['row_id'].apply(
    lambda x: '_'.join(x.split('_')[:-1]))
sample_sub['end_time'] = sample_sub['row_id'].apply(
    lambda x: int(x.split('_')[-1]))
n_ss  = sample_sub['soundscape'].nunique()
times = sorted(sample_sub['end_time'].unique())
print(f"Unique test soundscapes : {n_ss}")
print(f"Windows per soundscape  : {len(sample_sub)//n_ss}")
print(f"End-time values         : {times}")

# ── Ratings ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("10. RATINGS + COLLECTION")
print("=" * 60)
print(f"Rating distribution:\n{train['rating'].value_counts().sort_index()}")
print(f"\nCollection:\n{train['collection'].value_counts()}")


1. FILE SHAPES
train.csv          : (35549, 15)
taxonomy.csv       : (234, 5)
ss_labels.csv      : (1478, 4)
sample_submission  : (3, 235)

train.csv columns  : ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']

train.csv sample:
  primary_label secondary_labels type  latitude  longitude scientific_name   common_name class_name  inat_taxon_id         author   license  rating                                                           url                 filename collection
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0  https://static.inaturalist.org/sounds/1216197.mp3?1727581626  1161364/iNat1216197.ogg       iNat
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta  Guyalna cuta    Insecta        1161364  Lucas Barbos

In [2]:
# ============================================================
# BirdCLEF+ 2026 | Cell 2: Environment Setup + Model Scan
# ============================================================
import os, subprocess, sys
from pathlib import Path

BASE = Path('/kaggle/input/competitions/birdclef-2026')

# ── 1. Scan ALL available inputs ─────────────────────────────
print("=" * 60)
print("SCANNING ALL AVAILABLE INPUTS")
print("=" * 60)

for section in ['competitions', 'datasets']:
    p = Path(f'/kaggle/input/{section}')
    if p.exists():
        for item in sorted(p.iterdir()):
            print(f"/{section}/{item.name}/")
            if item.is_dir():
                for sub in sorted(item.iterdir())[:8]:
                    print(f"  {sub.name}")

# ── 2. Scan for any pre-loaded models ────────────────────────
print("\n" + "=" * 60)
print("SCANNING FOR PERCH / BIRD MODELS")
print("=" * 60)

model_search_paths = [
    '/kaggle/input',
    '/kaggle/input/datasets',
    '/kaggle/input/competitions',
]
found_models = []
for base in model_search_paths:
    for root, dirs, files in os.walk(base):
        for f in files:
            if any(kw in f.lower() for kw in
                   ['perch', 'bird', 'onnx', 'tflite', 'model', '.pb', 'efficientnet']):
                full = os.path.join(root, f)
                found_models.append(full)
                print(f"  FOUND: {full}")

if not found_models:
    print("  No pre-loaded models found — will download in Cell 3")

# ── 3. Check GPU/CPU availability ─────────────────────────────
print("\n" + "=" * 60)
print("COMPUTE ENVIRONMENT")
print("=" * 60)
import platform
print(f"Python  : {sys.version}")
print(f"OS      : {platform.platform()}")

try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    print(f"CUDA    : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
except ImportError:
    print("PyTorch : not installed")

try:
    import tensorflow as tf
    print(f"TF      : {tf.__version__}")
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TF GPUs : {gpus}")
except ImportError:
    print("TF      : not installed")

# ── 4. Check installed packages ───────────────────────────────
print("\n" + "=" * 60)
print("KEY PACKAGES")
print("=" * 60)
packages = ['librosa', 'numpy', 'pandas', 'sklearn',
            'onnxruntime', 'timm', 'torchaudio', 'soundfile']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f"  {pkg:20s}: {ver}")
    except ImportError:
        print(f"  {pkg:20s}: NOT INSTALLED")

# ── 5. Install missing packages ───────────────────────────────
print("\n" + "=" * 60)
print("INSTALLING MISSING PACKAGES")
print("=" * 60)
to_install = []
try:
    import onnxruntime
except ImportError:
    to_install.append('onnxruntime')
try:
    import soundfile
except ImportError:
    to_install.append('soundfile')
try:
    import timm
except ImportError:
    to_install.append('timm')

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + to_install)
    print("Done.")
else:
    print("All packages present.")

# ── 6. Quick audio load test ──────────────────────────────────
print("\n" + "=" * 60)
print("AUDIO LOAD TEST")
print("=" * 60)
import librosa, numpy as np

audio_files = list(BASE.rglob('train_audio/**/*.ogg'))[:3]
for fp in audio_files:
    y, sr = librosa.load(str(fp), sr=32000, duration=5.0, mono=True)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=128, fmin=50, fmax=14000,
        hop_length=320, n_fft=1024)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    print(f"  File : {fp.name}")
    print(f"  Audio: shape={y.shape}, sr={sr}")
    print(f"  Mel  : shape={mel_db.shape}, min={mel_db.min():.1f}, max={mel_db.max():.1f}")

# ── 7. Soundscape load test ───────────────────────────────────
print("\n" + "=" * 60)
print("SOUNDSCAPE LOAD TEST")
print("=" * 60)
ss_files = list((BASE / 'train_soundscapes').rglob('*.ogg'))[:2]
for fp in ss_files:
    y, sr = librosa.load(str(fp), sr=32000, mono=True)
    duration = len(y) / sr
    n_windows = int(duration // 5)
    print(f"  File     : {fp.name}")
    print(f"  Duration : {duration:.1f}s  →  {n_windows} × 5s windows")

# ── 8. Full config for rest of pipeline ───────────────────────
print("\n" + "=" * 60)
print("PIPELINE CONFIG (save these values)")
print("=" * 60)

CFG = {
    # Paths
    'BASE'          : str(BASE),
    'TRAIN_CSV'     : str(BASE / 'train.csv'),
    'TAXON_CSV'     : str(BASE / 'taxonomy.csv'),
    'SS_LABELS'     : str(BASE / 'train_soundscapes_labels.csv'),
    'TRAIN_AUDIO'   : str(BASE / 'train_audio'),
    'TRAIN_SS'      : str(BASE / 'train_soundscapes'),
    # Audio
    'SR'            : 32000,
    'WINDOW_SEC'    : 5,
    'N_MELS'        : 128,
    'FMIN'          : 50,
    'FMAX'          : 14000,
    'HOP_LENGTH'    : 320,
    'N_FFT'         : 1024,
    # Model
    'N_CLASSES'     : 234,
    'EMBED_DIM'     : 1280,      # Perch v2
    'MLP_HIDDEN'    : 512,
    'DROPOUT'       : 0.3,
    # Training
    'BATCH_SIZE'    : 32,
    'LR'            : 3e-4,
    'EPOCHS'        : 30,
    'LABEL_SMOOTH'  : 0.05,
    # Weighting
    'SOUNDSCAPE_WEIGHT': 3.0,    # labeled soundscapes get 3x weight
    'SONOTYPE_WEIGHT'  : 5.0,    # sonotypes only learnable from soundscapes
    # Runtime
    'SEED'          : 42,
}

for k, v in CFG.items():
    print(f"  {k:25s}: {v}")

print("\n" + "=" * 60)
print("=" * 60)

SCANNING ALL AVAILABLE INPUTS
/competitions/birdclef-2026/
  recording_location.txt
  sample_submission.csv
  taxonomy.csv
  test_soundscapes
  train.csv
  train_audio
  train_soundscapes
  train_soundscapes_labels.csv

SCANNING FOR PERCH / BIRD MODELS
  No pre-loaded models found — will download in Cell 3

COMPUTE ENVIRONMENT
Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
OS      : Linux-6.6.113+-x86_64-with-glibc2.35
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


2026-04-21 17:44:03.287887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776793443.731859      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776793443.846717      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776793444.878035      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776793444.878075      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776793444.878078      23 computation_placer.cc:177] computation placer alr

TF      : 2.19.0
TF GPUs : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

KEY PACKAGES
  librosa             : 0.11.0
  numpy               : 2.0.2
  pandas              : 2.3.3
  sklearn             : 1.6.1
  onnxruntime         : NOT INSTALLED
  timm                : 1.0.25
  torchaudio          : 2.10.0+cu128
  soundfile           : 0.13.1

INSTALLING MISSING PACKAGES
Installing: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.8 MB/s eta 0:00:00
Done.

AUDIO LOAD TEST
  File : XC252355.ogg
  Audio: shape=(160000,), sr=32000
  Mel  : shape=(128, 501), min=-80.0, max=0.0
  File : XC550621.ogg
  Audio: shape=(160000,), sr=32000
  Mel  : shape=(128, 501), min=-80.0, max=0.0
  File : XC172127.ogg
  Audio: shape=(142106,), sr=32000
  Mel  : shape=(128, 445), min=-80.0, max=0.0

SOUNDSCAPE LOAD TEST
  File     : BC2026_Train_2754_S02_20211024_210000.ogg
  Duration : 60.0s  →  

In [3]:
# Quick scan — what's in the tonylica dataset?
import os
from pathlib import Path

p = Path('/kaggle/input/datasets/tonylica')
print("tonylica contents:")
for root, dirs, files in os.walk(str(p)):
    level = root.replace(str(p), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:10]:
        size = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"{indent}  {f}  ({size:.1f} MB)")
    if len(files) > 10:
        print(f"{indent}  ... {len(files)-10} more files")

tonylica contents:


In [4]:
# ============================================================
# BirdCLEF+ 2026 | Cell 3: Download Perch + Verify GPU
# ============================================================
import os, sys, subprocess
import numpy as np
import torch
from pathlib import Path

# ── Verify GPU ───────────────────────────────────────────────
print("=" * 60)
print("GPU CHECK")
print("=" * 60)
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count      : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}        : {p.name}")
        print(f"  VRAM         : {p.total_memory/1e9:.1f} GB")
else:
    print("WARNING: GPU not detected — check Settings > Accelerator")

# ── Download Perch v2 via TF Hub ─────────────────────────────
print("\n" + "=" * 60)
print("DOWNLOADING PERCH V2 MODEL")
print("=" * 60)

# Install chirp (Google's bioacoustics library that wraps Perch)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tensorflow-hub', 'onnx', 'tf2onnx'], check=False)

import tensorflow as tf
import tensorflow_hub as hub

# Perch v2 CPU model on TF Hub
PERCH_URL = "https://tfhub.dev/google/bird-vocalization-classifier/4"

print(f"Loading from: {PERCH_URL}")
print("(This takes 2–3 minutes on first download...)")

try:
    perch_model = hub.load(PERCH_URL)
    print("Perch loaded successfully via TF Hub")
    
    # Test with dummy audio
    dummy = tf.zeros([1, 160000])  # 5s @ 32kHz
    out = perch_model.signatures['serving_default'](inputs=dummy)
    print(f"Output keys   : {list(out.keys())}")
    for k, v in out.items():
        print(f"  {k:20s}: shape={v.shape}, dtype={v.dtype}")
        
except Exception as e:
    print(f"TF Hub failed: {e}")
    print("Trying direct Kaggle model path...")
    
    # Fallback: try all possible Kaggle model paths
    possible = [
        '/kaggle/input/perch-v2-cpu',
        '/kaggle/input/models/google/bird-vocalization-classifier',
        '/kaggle/input/datasets/google/bird-vocalization-classifier',
        '/root/.cache/tfhub_modules',
    ]
    for p in possible:
        if os.path.exists(p):
            print(f"Found at: {p}")
            for root, dirs, files in os.walk(p):
                for f in files[:5]:
                    print(f"  {os.path.join(root,f)}")
        else:
            print(f"Not at  : {p}")

# ── Save Perch locally for reuse ────────────────────────────
print("\n" + "=" * 60)
print("SAVING MODEL TO /kaggle/working/")
print("=" * 60)
WORK = Path('/kaggle/working')
MODEL_DIR = WORK / 'perch_v2'
MODEL_DIR.mkdir(exist_ok=True)

try:
    tf.saved_model.save(perch_model, str(MODEL_DIR))
    size_mb = sum(
        os.path.getsize(os.path.join(r,f))
        for r,d,files in os.walk(MODEL_DIR)
        for f in files
    ) / 1e6
    print(f"Saved to : {MODEL_DIR}")
    print(f"Size     : {size_mb:.1f} MB")
except Exception as e:
    print(f"Save failed: {e} — will reload from hub each time")

# ── Extract embeddings test ──────────────────────────────────
print("\n" + "=" * 60)
print("EMBEDDING EXTRACTION TEST")
print("=" * 60)
import librosa

BASE = Path('/kaggle/input/competitions/birdclef-2026')
test_files = list((BASE / 'train_audio').rglob('*.ogg'))[:3]

def load_clip_5s(fp, sr=32000):
    y, _ = librosa.load(str(fp), sr=sr, duration=5.0, mono=True)
    if len(y) < sr * 5:
        y = np.pad(y, (0, sr*5 - len(y)))
    return y.astype(np.float32)

def get_embedding(model, audio_np):
    """Extract 1280-dim embedding from 5s audio clip"""
    inp = tf.constant(audio_np[np.newaxis, :])  # [1, 160000]
    out = model.signatures['serving_default'](inputs=inp)
    # embedding key varies by model version
    for key in ['embedding', 'embeddings', 'output_0']:
        if key in out:
            return out[key].numpy()[0]
    # fallback: return first output
    return list(out.values())[0].numpy()[0]

embeddings = []
for fp in test_files:
    audio = load_clip_5s(fp)
    emb   = get_embedding(perch_model, audio)
    embeddings.append(emb)
    print(f"  {fp.name[:40]:40s} → embedding shape: {emb.shape}")

embeddings = np.array(embeddings)
print(f"\nBatch embedding shape : {embeddings.shape}")
print(f"Embedding min/max     : {embeddings.min():.3f} / {embeddings.max():.3f}")
print(f"Embedding mean/std    : {embeddings.mean():.3f} / {embeddings.std():.3f}")



GPU CHECK
CUDA available : True
GPU count      : 2
  GPU 0        : Tesla T4
  VRAM         : 15.6 GB
  GPU 1        : Tesla T4
  VRAM         : 15.6 GB

DOWNLOADING PERCH V2 MODEL
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 14.5 MB/s eta 0:00:00
Loading from: https://tfhub.dev/google/bird-vocalization-classifier/4
(This takes 2–3 minutes on first download...)


I0000 00:00:1776793653.665073      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776793653.667957      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Perch loaded successfully via TF Hub


I0000 00:00:1776793658.274482      91 service.cc:152] XLA service 0x7ec2fc008e60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776793658.274540      91 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776793658.274545      91 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776793659.989793      91 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-21 17:47:46.776687: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 17:47:46.913894: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 17:47:47.092754: E external/local_xl

Output keys   : ['output_1', 'output_0']
  output_1            : shape=(1, 1280), dtype=<dtype: 'float32'>
  output_0            : shape=(1, 10932), dtype=<dtype: 'float32'>

SAVING MODEL TO /kaggle/working/
INFO:tensorflow:Assets written to: /kaggle/working/perch_v2/assets


INFO:tensorflow:Assets written to: /kaggle/working/perch_v2/assets


Saved to : /kaggle/working/perch_v2
Size     : 98.3 MB

EMBEDDING EXTRACTION TEST
  XC252355.ogg                             → embedding shape: (10932,)
  XC550621.ogg                             → embedding shape: (10932,)
  XC172127.ogg                             → embedding shape: (10932,)

Batch embedding shape : (3, 10932)
Embedding min/max     : -25.437 / 6.140
Embedding mean/std    : -15.799 / 2.697


In [5]:
# ============================================================
# BirdCLEF+ 2026 | Cell 4: Full Feature Extraction
# Extract Perch embeddings for ALL training data
# Runtime: ~25-35 min on T4 GPU
# ============================================================
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa, time, gc
from pathlib import Path
from tqdm import tqdm

BASE  = Path('/kaggle/input/competitions/birdclef-2026')
WORK  = Path('/kaggle/working')

# ── Reload Perch ─────────────────────────────────────────────
print("Loading Perch from saved model...")
perch_model = tf.saved_model.load(str(WORK / 'perch_v2'))
print("Loaded.")

# ── FIXED embedding function ─────────────────────────────────
def get_embedding(audio_np):
    """
    Returns 1280-dim embedding from output_1
    audio_np: float32 array of shape (160000,) = 5s @ 32kHz
    """
    inp = tf.constant(audio_np[np.newaxis, :])
    out = perch_model.signatures['serving_default'](inputs=inp)
    return out['output_1'].numpy()[0]  # (1280,)

# Quick sanity check
dummy = np.zeros(160000, dtype=np.float32)
emb   = get_embedding(dummy)
print(f"Embedding shape  : {emb.shape}")   # must be (1280,)
print(f"Embedding dtype  : {emb.dtype}")
assert emb.shape == (1280,), f"Wrong shape: {emb.shape}"
print("Embedding check PASSED")

# ── Audio loading helper ──────────────────────────────────────
SR         = 32000
WINDOW_LEN = SR * 5  # 160000 samples

def load_5s_clip(fp, offset=0.0):
    """Load 5s from fp starting at offset. Pad if short."""
    try:
        y, _ = librosa.load(str(fp), sr=SR, offset=offset,
                            duration=5.0, mono=True)
        if len(y) < WINDOW_LEN:
            y = np.pad(y, (0, WINDOW_LEN - len(y)))
        return y[:WINDOW_LEN].astype(np.float32)
    except Exception as e:
        return np.zeros(WINDOW_LEN, dtype=np.float32)

# ── PART A: Extract train_audio embeddings ───────────────────
print("\n" + "=" * 60)
print("PART A: train_audio embeddings (35,549 clips)")
print("=" * 60)

train    = pd.read_csv(BASE / 'train.csv')
taxon    = pd.read_csv(BASE / 'taxonomy.csv')
label2idx = {l: i for i, l in enumerate(taxon['primary_label'].tolist())}

BATCH = 64  # process in batches for speed
all_embs_A  = []
all_labels_A = []
all_weights_A = []

# Pre-compute per-species sample weights
label_counts = train['primary_label'].value_counts()

def get_weight(sp):
    n = label_counts.get(sp, 0)
    if n == 0:   return 5.0
    if n < 5:    return 4.0
    if n < 20:   return 2.0
    if n < 50:   return 1.5
    return 1.0

# Filter: skip rating < 1.0 for XC (but keep all iNat)
train_filtered = train[
    (train['collection'] == 'iNat') |
    (train['rating'] == 0) |      # unrated — keep
    (train['rating'] >= 1.0)      # rated 1+ — keep
].reset_index(drop=True)

print(f"Clips after quality filter: {len(train_filtered)}")

audio_dir = BASE / 'train_audio'
errors = 0
batch_audio, batch_meta = [], []

def flush_batch(batch_audio, batch_meta):
    if not batch_audio:
        return
    batch_np = np.stack(batch_audio)            # (B, 160000)
    batch_tf = tf.constant(batch_np)            # (B, 160000)
    out = perch_model.signatures['serving_default'](inputs=batch_tf)
    embs = out['output_1'].numpy()              # (B, 1280)
    for emb, (label, weight) in zip(embs, batch_meta):
        all_embs_A.append(emb)
        all_labels_A.append(label)
        all_weights_A.append(weight)

t0 = time.time()
for idx, row in tqdm(train_filtered.iterrows(),
                     total=len(train_filtered), desc="train_audio"):
    fp     = audio_dir / row['filename']
    label  = row['primary_label']
    weight = get_weight(label)
    audio  = load_5s_clip(fp, offset=0.0)

    batch_audio.append(audio)
    batch_meta.append((label, weight))

    if len(batch_audio) == BATCH:
        flush_batch(batch_audio, batch_meta)
        batch_audio, batch_meta = [], []

flush_batch(batch_audio, batch_meta)  # remaining

elapsed = time.time() - t0
print(f"\nPart A done: {len(all_embs_A)} embeddings in {elapsed/60:.1f} min")
print(f"Errors     : {errors}")

embs_A    = np.array(all_embs_A,   dtype=np.float32)
labels_A  = np.array(all_labels_A)
weights_A = np.array(all_weights_A, dtype=np.float32)

print(f"embs_A shape   : {embs_A.shape}")
print(f"labels sample  : {labels_A[:5]}")
print(f"weight dist    : {np.unique(weights_A, return_counts=True)}")

np.save(WORK / 'embs_train.npy',    embs_A)
np.save(WORK / 'labels_train.npy',  labels_A)
np.save(WORK / 'weights_train.npy', weights_A)
print("Saved Part A.")

# ── PART B: Labeled soundscape embeddings ────────────────────
print("\n" + "=" * 60)
print("PART B: labeled soundscape segments (1,478 segments)")
print("= " * 30)

ss_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
ss_dir    = BASE / 'train_soundscapes'

all_embs_B, all_labels_B, all_weights_B = [], [], []

# Sonotype list — get 5x weight
sonotypes = set(taxon[taxon['primary_label'].str.startswith('47158son')]['primary_label'])

batch_audio, batch_meta = [], []

for _, row in tqdm(ss_labels.iterrows(),
                   total=len(ss_labels), desc="soundscape_labels"):
    fp = ss_dir / row['filename']
    if not fp.exists():
        continue

    # Parse start time to offset in seconds
    start_str = str(row['start'])  # e.g. "00:00:05"
    parts = start_str.split(':')
    offset = int(parts[0])*3600 + int(parts[1])*60 + float(parts[2])

    audio = load_5s_clip(fp, offset=offset)

    # Multi-label: one entry per species in this window
    label_str = str(row['primary_label'])
    species_in_window = [s.strip() for s in label_str.split(';') if s.strip()]

    # Build multi-hot vector
    multi_hot = np.zeros(234, dtype=np.float32)
    max_weight = 1.0
    for sp in species_in_window:
        if sp in label2idx:
            multi_hot[label2idx[sp]] = 1.0
            w = 5.0 if sp in sonotypes else 3.0
            max_weight = max(max_weight, w)

    batch_audio.append(audio)
    batch_meta.append((multi_hot, max_weight))

    if len(batch_audio) == BATCH:
        # flush
        batch_np = np.stack(batch_audio)
        out = perch_model.signatures['serving_default'](
                  inputs=tf.constant(batch_np))
        embs = out['output_1'].numpy()
        for emb, (mh, wt) in zip(embs, batch_meta):
            all_embs_B.append(emb)
            all_labels_B.append(mh)
            all_weights_B.append(wt)
        batch_audio, batch_meta = [], []

# flush remaining
if batch_audio:
    batch_np = np.stack(batch_audio)
    out = perch_model.signatures['serving_default'](
              inputs=tf.constant(batch_np))
    embs = out['output_1'].numpy()
    for emb, (mh, wt) in zip(embs, batch_meta):
        all_embs_B.append(emb)
        all_labels_B.append(mh)
        all_weights_B.append(wt)

embs_B    = np.array(all_embs_B,   dtype=np.float32)
labels_B  = np.array(all_labels_B, dtype=np.float32)  # multi-hot (N, 234)
weights_B = np.array(all_weights_B, dtype=np.float32)

print(f"\nPart B done: {len(embs_B)} embeddings")
print(f"embs_B shape    : {embs_B.shape}")
print(f"labels_B shape  : {labels_B.shape}")
print(f"weight dist     : {np.unique(weights_B, return_counts=True)}")

np.save(WORK / 'embs_soundscape.npy',    embs_B)
np.save(WORK / 'labels_soundscape.npy',  labels_B)
np.save(WORK / 'weights_soundscape.npy', weights_B)
print("Saved Part B.")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)
total_samples = len(embs_A) + len(embs_B)
print(f"train_audio clips     : {len(embs_A)}")
print(f"soundscape segments   : {len(embs_B)}")
print(f"Total training samples: {total_samples}")
print(f"Embedding dim         : {embs_A.shape[1]}")

# Check disk usage
import os
for f in ['embs_train.npy', 'labels_train.npy', 'weights_train.npy',
          'embs_soundscape.npy', 'labels_soundscape.npy', 'weights_soundscape.npy']:
    sz = os.path.getsize(WORK / f) / 1e6
    print(f"  {f:35s}: {sz:.1f} MB")


Loading Perch from saved model...
Loaded.
Embedding shape  : (1280,)
Embedding dtype  : float32
Embedding check PASSED

PART A: train_audio embeddings (35,549 clips)
Clips after quality filter: 35527


train_audio:   0%|          | 59/35527 [00:01<12:47, 46.24it/s]2026-04-21 17:49:06.359551: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv.74 = (f32[64,640,501,1]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,1,160640,1]{3,2,1,0} %bitcast.8232, f32[640,1,640,1]{3,2,1,0} %bitcast.8233), window={size=640x1 stride=320x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", metadata={op_type="Conv2D" op_name="StatefulPartitionedCall/jax2tf_infer_fn_/TaxonomyModel/frontend/Conv2D" source_file="dummy_file_name" source_line=10}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-21 17:49:06.879409: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.520048869s
Trying algorithm eng0{} for conv %cud


Part A done: 35527 embeddings in 17.3 min
Errors     : 0
embs_A shape   : (35527, 1280)
labels sample  : ['1161364' '1161364' '1161364' '1161364' '1161364']
weight dist    : (array([1. , 1.5, 2. , 4. ], dtype=float32), array([34755,   521,   219,    32]))
Saved Part A.

PART B: labeled soundscape segments (1,478 segments)
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 


soundscape_labels: 100%|██████████| 1478/1478 [00:20<00:00, 71.66it/s]
2026-04-21 18:06:45.130698: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:06:45.270333: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:06:45.469702: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:06:45.609703: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsigh


Part B done: 1478 embeddings
embs_B shape    : (1478, 1280)
labels_B shape  : (1478, 234)
weight dist     : (array([3., 5.], dtype=float32), array([1142,  336]))
Saved Part B.

EXTRACTION SUMMARY
train_audio clips     : 35527
soundscape segments   : 1478
Total training samples: 37005
Embedding dim         : 1280
  embs_train.npy                     : 181.9 MB
  labels_train.npy                   : 1.0 MB
  weights_train.npy                  : 0.1 MB
  embs_soundscape.npy                : 7.6 MB
  labels_soundscape.npy              : 1.4 MB
  weights_soundscape.npy             : 0.0 MB


In [6]:
# ============================================================
# BirdCLEF+ 2026 | Cell 5: MLP Training
# Train classification head on Perch embeddings
# Runtime: ~15-20 min on T4 GPU
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import time, gc

WORK = Path('/kaggle/working')
BASE = Path('/kaggle/input/competitions/birdclef-2026')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Load embeddings ──────────────────────────────────────────
print("Loading embeddings...")
embs_A    = np.load(WORK / 'embs_train.npy')
labels_A  = np.load(WORK / 'labels_train.npy')
weights_A = np.load(WORK / 'weights_train.npy')
embs_B    = np.load(WORK / 'embs_soundscape.npy')
labels_B  = np.load(WORK / 'labels_soundscape.npy')   # multi-hot (1478, 234)
weights_B = np.load(WORK / 'weights_soundscape.npy')

# ── Load taxonomy — build label encoder ─────────────────────
taxon     = pd.read_csv(BASE / 'taxonomy.csv')
ALL_SPECIES = taxon['primary_label'].tolist()          # 234 species, fixed order
N_CLASSES   = len(ALL_SPECIES)
sp2idx      = {sp: i for i, sp in enumerate(ALL_SPECIES)}
print(f"Classes: {N_CLASSES}")

# ── Convert Part A labels to multi-hot ───────────────────────
print("Converting Part A labels to multi-hot...")
labels_A_mh = np.zeros((len(labels_A), N_CLASSES), dtype=np.float32)
for i, sp in enumerate(labels_A):
    if sp in sp2idx:
        labels_A_mh[i, sp2idx[sp]] = 1.0

# ── Combine A + B ────────────────────────────────────────────
embs_all    = np.vstack([embs_A,    embs_B])
labels_all  = np.vstack([labels_A_mh, labels_B])
weights_all = np.concatenate([weights_A, weights_B])

print(f"Combined: {embs_all.shape[0]} samples, {N_CLASSES} classes")
print(f"Weight distribution: {np.unique(weights_all, return_counts=True)}")

# ── Normalize embeddings ─────────────────────────────────────
emb_mean = embs_all.mean(axis=0)
emb_std  = embs_all.std(axis=0) + 1e-8
embs_norm = (embs_all - emb_mean) / emb_std
np.save(WORK / 'emb_mean.npy', emb_mean)
np.save(WORK / 'emb_std.npy',  emb_std)
print("Embeddings normalized and stats saved.")

# ── Dataset ──────────────────────────────────────────────────
class EmbDataset(Dataset):
    def __init__(self, embs, labels, weights):
        self.embs    = torch.tensor(embs,    dtype=torch.float32)
        self.labels  = torch.tensor(labels,  dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self): return len(self.embs)

    def __getitem__(self, i):
        return self.embs[i], self.labels[i], self.weights[i]

# ── MLP Model ────────────────────────────────────────────────
class BirdMLP(nn.Module):
    def __init__(self, in_dim=1280, hidden=512, n_classes=234, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x):
        return self.net(x)   # raw logits

# ── Weighted BCE loss ─────────────────────────────────────────
def weighted_bce(logits, targets, sample_weights, label_smooth=0.05):
    # Label smoothing
    targets_smooth = targets * (1 - label_smooth) + 0.5 * label_smooth
    loss = F.binary_cross_entropy_with_logits(
        logits, targets_smooth, reduction='none')        # (B, C)
    loss = loss.mean(dim=1)                              # (B,)
    loss = (loss * sample_weights).mean()
    return loss

# ── Train / eval split (stratified on primary class) ─────────
# Use Part A indices for stratification (single-label)
primary_class = labels_A_mh.argmax(axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(embs_A, primary_class))

# Validation = Part A val fold only (clean single-label for AUC calc)
val_embs   = embs_norm[val_idx]
val_labels = labels_all[val_idx]

# Training = Part A train fold + ALL of Part B
train_idx_full = np.concatenate([
    train_idx,
    np.arange(len(embs_A), len(embs_all))   # Part B indices
])
train_embs    = embs_norm[train_idx_full]
train_labels  = labels_all[train_idx_full]
train_weights = weights_all[train_idx_full]

print(f"\nTrain samples : {len(train_embs)}")
print(f"Val samples   : {len(val_embs)}")

train_ds = EmbDataset(train_embs,  train_labels,  train_weights)
val_ds   = EmbDataset(val_embs,    val_labels,
                      np.ones(len(val_embs), dtype=np.float32))

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

# ── Training loop ─────────────────────────────────────────────
model     = BirdMLP(in_dim=1280, hidden=512,
                    n_classes=N_CLASSES, dropout=0.3).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=30, eta_min=1e-6)

EPOCHS       = 30
best_auc     = 0.0
best_epoch   = 0
patience     = 7
no_improve   = 0

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for emb, lbl, wt in train_loader:
        emb, lbl, wt = emb.to(DEVICE), lbl.to(DEVICE), wt.to(DEVICE)
        optimizer.zero_grad()
        logits = model(emb)
        loss   = weighted_bce(logits, lbl, wt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for emb, lbl, _ in val_loader:
            emb = emb.to(DEVICE)
            prob = torch.sigmoid(model(emb)).cpu().numpy()
            all_probs.append(prob)
            all_true.append(lbl.numpy())

    probs = np.vstack(all_probs)
    trues = np.vstack(all_true)

    # Macro AUC — skip classes with no positives in val set
    aucs = []
    for c in range(N_CLASSES):
        if trues[:, c].sum() > 0:
            aucs.append(roc_auc_score(trues[:, c], probs[:, c]))
    val_auc = np.mean(aucs) if aucs else 0.0

    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"loss={train_loss:.4f} | "
          f"val_auc={val_auc:.4f} | "
          f"classes_scored={len(aucs)} | "
          f"lr={scheduler.get_last_lr()[0]:.2e} | "
          f"{elapsed:.0f}s")

    if val_auc > best_auc:
        best_auc   = val_auc
        best_epoch = epoch
        no_improve = 0
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'val_auc'    : val_auc,
            'sp2idx'     : sp2idx,
        }, WORK / 'best_mlp.pt')
        print(f"  *** New best: {best_auc:.4f} — saved ***")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stop at epoch {epoch}")
            break

print("\n" + "=" * 60)
print(f"TRAINING COMPLETE")
print(f"Best AUC   : {best_auc:.4f}  (epoch {best_epoch})")
print("=" * 60)

# ── Per-class AUC breakdown ───────────────────────────────────
print("\nPer-class AUC (bottom 20 — hardest species):")
class_aucs = {}
for c in range(N_CLASSES):
    if trues[:, c].sum() > 0:
        class_aucs[ALL_SPECIES[c]] = roc_auc_score(trues[:, c], probs[:, c])

sorted_aucs = sorted(class_aucs.items(), key=lambda x: x[1])
for sp, auc in sorted_aucs[:20]:
    row = taxon[taxon['primary_label'] == sp]
    cls = row['class_name'].values[0] if len(row) > 0 else '?'
    print(f"  {sp:25s} ({cls:10s}): {auc:.4f}")

print(f"\nSaved: {WORK / 'best_mlp.pt'}")

Device: cuda:0
Loading embeddings...
Classes: 234
Converting Part A labels to multi-hot...
Combined: 37005 samples, 234 classes
Weight distribution: (array([1. , 1.5, 2. , 3. , 4. , 5. ], dtype=float32), array([34755,   521,   219,  1142,    32,   336]))
Embeddings normalized and stats saved.

Train samples : 29899
Val samples   : 7106

TRAINING
Epoch 01/30 | loss=0.5299 | val_auc=0.6436 | classes_scored=198 | lr=2.99e-04 | 3s
  *** New best: 0.6436 — saved ***
Epoch 02/30 | loss=0.2617 | val_auc=0.7020 | classes_scored=198 | lr=2.97e-04 | 2s
  *** New best: 0.7020 — saved ***
Epoch 03/30 | loss=0.1861 | val_auc=0.7460 | classes_scored=198 | lr=2.93e-04 | 2s
  *** New best: 0.7460 — saved ***
Epoch 04/30 | loss=0.1631 | val_auc=0.7852 | classes_scored=198 | lr=2.87e-04 | 2s
  *** New best: 0.7852 — saved ***
Epoch 05/30 | loss=0.1548 | val_auc=0.8133 | classes_scored=198 | lr=2.80e-04 | 2s
  *** New best: 0.8133 — saved ***
Epoch 06/30 | loss=0.1513 | val_auc=0.8468 | classes_scored=19

In [7]:
# ============================================================
# BirdCLEF+ 2026 | Cell 6: Pseudo-label Distillation Round 1
# Generate soft labels on ALL unlabeled soundscapes
# Then retrain MLP with expanded dataset
# Runtime: ~35-40 min total
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tensorflow as tf
import librosa, time, gc
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

WORK   = Path('/kaggle/working')
BASE   = Path('/kaggle/input/competitions/birdclef-2026')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

taxon       = pd.read_csv(BASE / 'taxonomy.csv')
ALL_SPECIES = taxon['primary_label'].tolist()
N_CLASSES   = len(ALL_SPECIES)
sp2idx      = {sp: i for i, sp in enumerate(ALL_SPECIES)}

# ── Reload MLP ───────────────────────────────────────────────
class BirdMLP(nn.Module):
    def __init__(self, in_dim=1280, hidden=512, n_classes=234, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x): return self.net(x)

ckpt = torch.load(WORK / 'best_mlp.pt', map_location=DEVICE, weights_only=False)
model = BirdMLP().to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded MLP — best val AUC was {ckpt['val_auc']:.4f}")

# ── Reload Perch + normalization stats ───────────────────────
print("Loading Perch...")
perch_model = tf.saved_model.load(str(WORK / 'perch_v2'))
emb_mean    = np.load(WORK / 'emb_mean.npy')
emb_std     = np.load(WORK / 'emb_std.npy')

SR         = 32000
WINDOW_LEN = SR * 5

def load_5s(fp, offset=0.0):
    try:
        y, _ = librosa.load(str(fp), sr=SR, offset=offset,
                            duration=5.0, mono=True)
        if len(y) < WINDOW_LEN:
            y = np.pad(y, (0, WINDOW_LEN - len(y)))
        return y[:WINDOW_LEN].astype(np.float32)
    except:
        return np.zeros(WINDOW_LEN, dtype=np.float32)

def extract_emb_batch(audio_list):
    batch = tf.constant(np.stack(audio_list))
    out   = perch_model.signatures['serving_default'](inputs=batch)
    embs  = out['output_1'].numpy()                       # (B, 1280)
    return (embs - emb_mean) / emb_std                   # normalized

def predict_probs(embs_norm_np):
    """Run MLP on normalized embeddings, return sigmoid probs"""
    t = torch.tensor(embs_norm_np, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        logits = model(t)
        probs  = torch.sigmoid(logits).cpu().numpy()
    return probs

# ── Generate pseudo-labels on ALL train_soundscapes ──────────
print("\n" + "=" * 60)
print("STEP 1: Generating pseudo-labels on 10,658 soundscapes")
print("Each soundscape = 12 windows × 5s")
print("=" * 60)

ss_dir   = BASE / 'train_soundscapes'
ss_files = sorted(ss_dir.glob('*.ogg'))
print(f"Soundscape files found: {len(ss_files)}")

TEMPERATURE  = 0.7   # sharpen soft labels slightly
MIN_CONF     = 0.15  # only keep windows where at least 1 species > threshold
BATCH_SIZE   = 64

pseudo_embs, pseudo_labels, pseudo_weights = [], [], []
n_windows_total  = 0
n_windows_kept   = 0

# Check which files have gold labels (skip generating pseudo for those)
ss_label_df   = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
labeled_files = set(ss_label_df['filename'].unique())
print(f"Files with gold labels (skip pseudo): {len(labeled_files)}")

t0 = time.time()
batch_audio, batch_info = [], []

def flush_pseudo_batch(batch_audio, batch_info):
    global n_windows_total, n_windows_kept
    if not batch_audio: return
    embs_norm = extract_emb_batch(batch_audio)
    probs     = predict_probs(embs_norm)              # (B, 234)

    # Soft label with temperature sharpening
    # probs^(1/T) then renorm — makes confident predictions sharper
    sharp = np.power(np.clip(probs, 1e-7, 1-1e-7), 1.0 / TEMPERATURE)

    for i, (fname, offset) in enumerate(batch_info):
        n_windows_total += 1
        max_conf = sharp[i].max()
        if max_conf < MIN_CONF:
            continue                                  # skip silent windows
        n_windows_kept += 1
        pseudo_embs.append(embs_norm[i])
        pseudo_labels.append(sharp[i].astype(np.float32))
        pseudo_weights.append(1.0)                    # pseudo = weight 1.0

for fp in tqdm(ss_files, desc="soundscapes"):
    fname = fp.name
    if fname in labeled_files:
        continue   # already have gold labels for this file

    # 12 windows per 60s soundscape
    for w in range(12):
        offset = w * 5.0
        audio  = load_5s(fp, offset=offset)
        batch_audio.append(audio)
        batch_info.append((fname, offset))

        if len(batch_audio) == BATCH_SIZE:
            flush_pseudo_batch(batch_audio, batch_info)
            batch_audio, batch_info = [], []

flush_pseudo_batch(batch_audio, batch_info)

elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min")
print(f"Total windows   : {n_windows_total:,}")
print(f"Windows kept    : {n_windows_kept:,}  "
      f"({100*n_windows_kept/max(n_windows_total,1):.1f}%)")
print(f"Discarded silent: {n_windows_total - n_windows_kept:,}")

pseudo_embs_np    = np.array(pseudo_embs,    dtype=np.float32)
pseudo_labels_np  = np.array(pseudo_labels,  dtype=np.float32)
pseudo_weights_np = np.array(pseudo_weights, dtype=np.float32)

print(f"\npseudo_embs shape  : {pseudo_embs_np.shape}")
print(f"pseudo_label range : {pseudo_labels_np.min():.3f} – "
      f"{pseudo_labels_np.max():.3f}")

np.save(WORK / 'pseudo_embs.npy',    pseudo_embs_np)
np.save(WORK / 'pseudo_labels.npy',  pseudo_labels_np)
np.save(WORK / 'pseudo_weights.npy', pseudo_weights_np)
print("Pseudo-labels saved.")

# ── STEP 2: Retrain MLP with original + pseudo data ──────────
print("\n" + "=" * 60)
print("STEP 2: Retraining MLP with pseudo-labels added")
print("=" * 60)

# Reload original embeddings
embs_A    = np.load(WORK / 'embs_train.npy')
labels_A  = np.load(WORK / 'labels_train.npy')
weights_A = np.load(WORK / 'weights_train.npy')
embs_B    = np.load(WORK / 'embs_soundscape.npy')
labels_B  = np.load(WORK / 'labels_soundscape.npy')
weights_B = np.load(WORK / 'weights_soundscape.npy')

# Convert Part A to multi-hot
labels_A_mh = np.zeros((len(labels_A), N_CLASSES), dtype=np.float32)
for i, sp in enumerate(labels_A):
    if sp in sp2idx:
        labels_A_mh[i, sp2idx[sp]] = 1.0

# Normalize Part A embeddings
embs_A_norm = (embs_A - emb_mean) / emb_std

# Combine: original A + soundscape gold B + pseudo C
embs_all    = np.vstack([embs_A_norm, embs_B, pseudo_embs_np])
labels_all  = np.vstack([labels_A_mh, labels_B, pseudo_labels_np])
weights_all = np.concatenate([weights_A, weights_B, pseudo_weights_np])

print(f"Original train_audio : {len(embs_A_norm):,}")
print(f"Soundscape gold      : {len(embs_B):,}")
print(f"Pseudo-labeled       : {len(pseudo_embs_np):,}")
print(f"Total combined       : {len(embs_all):,}")

# Same val split as before
from sklearn.model_selection import StratifiedKFold
primary_class = labels_A_mh.argmax(axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_A, val_idx = next(skf.split(embs_A_norm, primary_class))

val_embs   = embs_A_norm[val_idx]
val_labels = labels_A_mh[val_idx]

# Training = A train fold + B + pseudo
train_idx_full = np.concatenate([
    train_idx_A,
    np.arange(len(embs_A_norm), len(embs_all))
])
train_embs    = embs_all[train_idx_full]
train_labels  = labels_all[train_idx_full]
train_weights = weights_all[train_idx_full]

print(f"Train : {len(train_embs):,}   Val : {len(val_embs):,}")

class EmbDataset(Dataset):
    def __init__(self, e, l, w):
        self.e = torch.tensor(e, dtype=torch.float32)
        self.l = torch.tensor(l, dtype=torch.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self): return len(self.e)
    def __getitem__(self, i): return self.e[i], self.l[i], self.w[i]

def weighted_bce(logits, targets, weights, label_smooth=0.05):
    t = targets * (1 - label_smooth) + 0.5 * label_smooth
    loss = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
    return (loss.mean(dim=1) * weights).mean()

train_ds     = EmbDataset(train_embs, train_labels, train_weights)
val_ds       = EmbDataset(val_embs,   val_labels,
                          np.ones(len(val_embs), dtype=np.float32))
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

# Fresh model for round 2
model2     = BirdMLP().to(DEVICE)
optimizer  = torch.optim.AdamW(model2.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
                 optimizer, T_max=25, eta_min=1e-6)

best_auc2, best_ep2, no_imp = 0.0, 0, 0

print("\nTraining round 2 (distilled)...")
for epoch in range(1, 26):
    model2.train()
    tr_loss = 0.0
    for e, l, w in train_loader:
        e, l, w = e.to(DEVICE), l.to(DEVICE), w.to(DEVICE)
        optimizer.zero_grad()
        loss = weighted_bce(model2(e), l, w)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item()
    scheduler.step()
    tr_loss /= len(train_loader)

    model2.eval()
    probs_v, true_v = [], []
    with torch.no_grad():
        for e, l, _ in val_loader:
            probs_v.append(torch.sigmoid(model2(e.to(DEVICE))).cpu().numpy())
            true_v.append(l.numpy())
    probs_v = np.vstack(probs_v)
    true_v  = np.vstack(true_v)

    aucs = [roc_auc_score(true_v[:,c], probs_v[:,c])
            for c in range(N_CLASSES) if true_v[:,c].sum() > 0]
    val_auc = np.mean(aucs)

    flag = ''
    if val_auc > best_auc2:
        best_auc2, best_ep2, no_imp = val_auc, epoch, 0
        torch.save({'epoch': epoch, 'model_state': model2.state_dict(),
                    'val_auc': val_auc, 'sp2idx': sp2idx},
                   WORK / 'best_mlp_distill.pt')
        flag = ' *** NEW BEST — saved ***'
    else:
        no_imp += 1
        if no_imp >= 6: print(f"Early stop at {epoch}"); break

    print(f"Ep {epoch:02d}/25 | loss={tr_loss:.4f} | "
          f"auc={val_auc:.4f} | lr={scheduler.get_last_lr()[0]:.2e}{flag}")

print("\n" + "=" * 60)
print(f"Round 1 best AUC : {ckpt['val_auc']:.4f}")
print(f"Round 2 best AUC : {best_auc2:.4f}  "
      f"(+{best_auc2 - ckpt['val_auc']:.4f})")
print("=" * 60)

Loaded MLP — best val AUC was 0.9375
Loading Perch...

STEP 1: Generating pseudo-labels on 10,658 soundscapes
Each soundscape = 12 windows × 5s
Soundscape files found: 10658
Files with gold labels (skip pseudo): 66


soundscapes: 100%|██████████| 10658/10658 [29:18<00:00,  6.06it/s]



Done in 29.3 min
Total windows   : 127,104
Windows kept    : 104,347  (82.1%)
Discarded silent: 22,757

pseudo_embs shape  : (104347, 1280)
pseudo_label range : 0.001 – 0.990
Pseudo-labels saved.

STEP 2: Retraining MLP with pseudo-labels added
Original train_audio : 35,527
Soundscape gold      : 1,478
Pseudo-labeled       : 104,347
Total combined       : 141,352
Train : 134,246   Val : 7,106

Training round 2 (distilled)...
Ep 01/25 | loss=0.3118 | auc=0.5695 | lr=1.99e-04 *** NEW BEST — saved ***
Ep 02/25 | loss=0.1724 | auc=0.6376 | lr=1.97e-04 *** NEW BEST — saved ***
Ep 03/25 | loss=0.1695 | auc=0.7254 | lr=1.93e-04 *** NEW BEST — saved ***
Ep 04/25 | loss=0.1689 | auc=0.8003 | lr=1.88e-04 *** NEW BEST — saved ***
Ep 05/25 | loss=0.1685 | auc=0.8412 | lr=1.81e-04 *** NEW BEST — saved ***
Ep 06/25 | loss=0.1681 | auc=0.8661 | lr=1.73e-04 *** NEW BEST — saved ***
Ep 07/25 | loss=0.1678 | auc=0.8935 | lr=1.64e-04 *** NEW BEST — saved ***
Ep 08/25 | loss=0.1675 | auc=0.9020 | lr=1.54

In [8]:
# ============================================================
# BirdCLEF+ 2026 | Cell 7: Inference + Submission
# CPU-compatible inference pipeline
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tensorflow as tf
import librosa, time, os
from pathlib import Path
from tqdm import tqdm

WORK  = Path('/kaggle/working')
BASE  = Path('/kaggle/input/competitions/birdclef-2026')
DEVICE = torch.device('cpu')  # submission must run on CPU

# ── Model definition ─────────────────────────────────────────
class BirdMLP(nn.Module):
    def __init__(self, in_dim=1280, hidden=512, n_classes=234, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x): return self.net(x)

# ── Load everything ──────────────────────────────────────────
print("Loading models and stats...")
taxon       = pd.read_csv(BASE / 'taxonomy.csv')
ALL_SPECIES = taxon['primary_label'].tolist()
N_CLASSES   = len(ALL_SPECIES)

emb_mean = np.load(WORK / 'emb_mean.npy')
emb_std  = np.load(WORK / 'emb_std.npy')

# Load both models — we'll blend them
ckpt1  = torch.load(WORK / 'best_mlp.pt',
                    map_location=DEVICE, weights_only=False)
ckpt2  = torch.load(WORK / 'best_mlp_distill.pt',
                    map_location=DEVICE, weights_only=False)

model1 = BirdMLP().to(DEVICE)
model1.load_state_dict(ckpt1['model_state'])
model1.eval()

model2 = BirdMLP().to(DEVICE)
model2.load_state_dict(ckpt2['model_state'])
model2.eval()

print(f"Model 1 val AUC: {ckpt1['val_auc']:.4f}")
print(f"Model 2 val AUC: {ckpt2['val_auc']:.4f}")

# Load Perch (CPU inference for submission)
print("Loading Perch for CPU inference...")
perch_model = tf.saved_model.load(str(WORK / 'perch_v2'))
print("Ready.")

# ── Audio helper ─────────────────────────────────────────────
SR         = 32000
WINDOW_LEN = SR * 5

def load_5s(fp, offset=0.0):
    try:
        y, _ = librosa.load(str(fp), sr=SR, offset=offset,
                            duration=5.0, mono=True)
        if len(y) < WINDOW_LEN:
            y = np.pad(y, (0, WINDOW_LEN - len(y)))
        return y[:WINDOW_LEN].astype(np.float32)
    except:
        return np.zeros(WINDOW_LEN, dtype=np.float32)

def get_emb_batch(audio_list):
    batch = tf.constant(np.stack(audio_list))
    out   = perch_model.signatures['serving_default'](inputs=batch)
    embs  = out['output_1'].numpy()
    return (embs - emb_mean) / emb_std

def predict_blend(embs_norm):
    t = torch.tensor(embs_norm, dtype=torch.float32)
    with torch.no_grad():
        p1 = torch.sigmoid(model1(t)).numpy()
        p2 = torch.sigmoid(model2(t)).numpy()
    # Blend: 50/50 since AUCs are nearly identical
    return 0.5 * p1 + 0.5 * p2

# ── Build species prior from pseudo-labels ───────────────────
print("\nBuilding species prior from pseudo-labels...")
pseudo_labels = np.load(WORK / 'pseudo_labels.npy')  # (104347, 234)

# Prior = mean probability per species across all soundscape windows
prior = pseudo_labels.mean(axis=0)  # (234,)
prior = prior / (prior.max() + 1e-8)  # normalize to [0,1]

print(f"Prior range: {prior.min():.4f} – {prior.max():.4f}")
print(f"Top 5 species by prior:")
top5 = np.argsort(prior)[::-1][:5]
for i in top5:
    print(f"  {ALL_SPECIES[i]:25s}: {prior[i]:.4f}")

# ── Run inference on test soundscapes ────────────────────────
print("\n" + "=" * 60)
print("RUNNING TEST INFERENCE")
print("=" * 60)

test_dir   = BASE / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

print(f"Test files found: {len(test_files)}")
print(f"Sample submission rows: {len(sample_sub)}")

# Parse row_ids to understand structure
sample_sub['soundscape'] = sample_sub['row_id'].apply(
    lambda x: '_'.join(x.split('_')[:-1]))
sample_sub['end_time']   = sample_sub['row_id'].apply(
    lambda x: int(x.split('_')[-1]))

# Build results dict
results = {}  # row_id → prob array (234,)

BATCH_SIZE = 32
t0         = time.time()

for fp in tqdm(test_files, desc="test inference"):
    fname    = fp.stem  # filename without .ogg
    duration = 60.0     # all test files are 60s

    # Collect all 12 windows
    batch_audio, batch_offsets = [], []
    for w in range(12):
        offset = w * 5.0
        batch_audio.append(load_5s(fp, offset))
        batch_offsets.append(offset)

    # Extract embeddings in one batch
    embs_norm = get_emb_batch(batch_audio)         # (12, 1280)
    probs     = predict_blend(embs_norm)            # (12, 234)

    # Apply prior calibration (mild — alpha=0.1)
    ALPHA = 0.1
    probs_cal = probs * (1 - ALPHA) + prior * ALPHA

    # Store per-window predictions
    for w, (offset, prob) in enumerate(zip(batch_offsets, probs_cal)):
        end_time = int(offset + 5)
        row_id   = f"{fname}_{end_time}"
        results[row_id] = prob

elapsed = time.time() - t0
print(f"\nInference done in {elapsed/60:.1f} min")
print(f"Predictions generated: {len(results)}")

# ── Build submission dataframe ────────────────────────────────
print("\nBuilding submission CSV...")

# Use sample_submission to get correct row order + columns
sub_cols = [c for c in sample_sub.columns if c != 'row_id'
            and c not in ['soundscape', 'end_time']]

rows = []
for _, row in sample_sub.iterrows():
    rid = row['row_id']
    if rid in results:
        prob = results[rid]
    else:
        # Fallback: use prior
        prob = prior.copy()
    rows.append([rid] + prob.tolist())

submission = pd.DataFrame(rows, columns=['row_id'] + ALL_SPECIES)

# Verify shape matches sample submission
print(f"Submission shape : {submission.shape}")
print(f"Expected columns : {len(sample_sub.columns)}")
print(f"Sample rows:")
print(submission.head(3).iloc[:, :6].to_string())

# Check for NaN/Inf
assert not submission.isnull().any().any(), "NaN found in submission!"
assert (submission[ALL_SPECIES] >= 0).all().all(), "Negative probs!"
assert (submission[ALL_SPECIES] <= 1).all().all(), "Probs > 1!"

# ── Save submission ───────────────────────────────────────────
sub_path = WORK / 'submission.csv'
submission.to_csv(sub_path, index=False)

size_kb = os.path.getsize(sub_path) / 1024
print(f"\nSaved: {sub_path}  ({size_kb:.1f} KB)")
print(f"Rows  : {len(submission)}")
print(f"Cols  : {len(submission.columns)}")

# ── Runtime estimate for real test ───────────────────────────
print("\n" + "=" * 60)
print("RUNTIME ESTIMATE")
print("=" * 60)
files_tested   = len(test_files)
time_per_file  = elapsed / max(files_tested, 1)
real_test_size = 600  # ~600 files in hidden test
est_runtime    = time_per_file * real_test_size / 60
print(f"Files tested        : {files_tested}")
print(f"Time per file       : {time_per_file:.2f}s")
print(f"Estimated for ~600  : {est_runtime:.1f} min")
print(f"Kaggle CPU limit    : 90 min")
print(f"Safety margin       : {90 - est_runtime:.1f} min")

print("\n" + "=" * 60)
print("Cell 7 COMPLETE — ready to submit!")
print("Submit via: Notebook > Submit > Commit & Run")
print("=" * 60)

Loading models and stats...
Model 1 val AUC: 0.9375
Model 2 val AUC: 0.9378
Loading Perch for CPU inference...
Ready.

Building species prior from pseudo-labels...
Prior range: 0.0303 – 1.0000
Top 5 species by prior:
  65380                    : 1.0000
  517063                   : 0.8625
  22973                    : 0.7045
  23158                    : 0.4838
  555146                   : 0.4637

RUNNING TEST INFERENCE
Test files found: 0
Sample submission rows: 3


test inference: 0it [00:00, ?it/s]


Inference done in 0.0 min
Predictions generated: 0

Building submission CSV...
Submission shape : (3, 235)
Expected columns : 237
Sample rows:
                                    row_id  1161364    116570   1176823   1491113   1595929
0   BC2026_Test_0001_S05_20250227_010002_5  0.03926  0.050529  0.041496  0.197234  0.042065
1  BC2026_Test_0001_S05_20250227_010002_10  0.03926  0.050529  0.041496  0.197234  0.042065
2  BC2026_Test_0001_S05_20250227_010002_15  0.03926  0.050529  0.041496  0.197234  0.042065

Saved: /kaggle/working/submission.csv  (15.8 KB)
Rows  : 3
Cols  : 235

RUNTIME ESTIMATE
Files tested        : 0
Time per file       : 0.00s
Estimated for ~600  : 0.0 min
Kaggle CPU limit    : 90 min
Safety margin       : 90.0 min

Cell 7 COMPLETE — ready to submit!
Submit via: Notebook > Submit > Commit & Run


In [9]:
# ============================================================
# Cell 8: Verify + Fix Submission Format
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

WORK = Path('/kaggle/working')
BASE = Path('/kaggle/input/competitions/birdclef-2026')

sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
our_sub    = pd.read_csv(WORK / 'submission.csv')
taxon      = pd.read_csv(BASE / 'taxonomy.csv')

print("Sample submission columns:", len(sample_sub.columns))
print("Our submission columns   :", len(our_sub.columns))

# Get expected species columns from sample_submission
expected_cols = [c for c in sample_sub.columns if c != 'row_id']
our_cols      = [c for c in our_sub.columns if c != 'row_id']

print(f"\nExpected species: {len(expected_cols)}")
print(f"Our species     : {len(our_cols)}")

missing  = set(expected_cols) - set(our_cols)
extra    = set(our_cols) - set(expected_cols)
print(f"Missing in ours : {len(missing)} → {sorted(missing)[:10]}")
print(f"Extra in ours   : {len(extra)}   → {sorted(extra)[:10]}")

# Check sample_sub row_id format
print(f"\nSample row_ids:")
for r in sample_sub['row_id'].tolist():
    print(f"  {r}")

Sample submission columns: 235
Our submission columns   : 235

Expected species: 234
Our species     : 234
Missing in ours : 0 → []
Extra in ours   : 0   → []

Sample row_ids:
  BC2026_Test_0001_S05_20250227_010002_5
  BC2026_Test_0001_S05_20250227_010002_10
  BC2026_Test_0001_S05_20250227_010002_15
